**Author:** Christopher Millward<br>
**Date:** August 30, 2026

The purpose of this notebook is to audit the computation time of each function call in my pipeline. This will help me understand where (if any) I can try to spend time optimizing.

In [1]:
import os 
import sys

# set root folder to project root
root_path = os.path.abspath(os.path.join(".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# auto reload
%load_ext autoreload
%autoreload 2

In [2]:
import cProfile
import pstats

from modules.data_loading import load_participant_details, load_motion_capture_data
from modules.data_preprocessing import clean_and_validate_data  
from modules.kinematics import calculate_bin_rotations

from config import RAW_PARTICIPANT_DETAILS_PATH

participant_details = load_participant_details(RAW_PARTICIPANT_DETAILS_PATH)
desired_filename = '1_R_MATRICES_NORM 5-29-2015' #largest file
participant = [p for p in participant_details if p.filename == desired_filename][0]

In [ ]:
with cProfile.Profile() as pr:
    # load the data
    raw_data = load_motion_capture_data(participant.filename)

    # clean and validate data
    clean_data = clean_and_validate_data(raw_data)

    # run kinematics
    right_arm_kinematics = calculate_bin_rotations(clean_data, 'right')
    left_arm_kinematics = calculate_bin_rotations(clean_data, 'left')

    # save heatmap data
    participant.right.humerothoracic.heatmap = right_arm_kinematics
    participant.left.humerothoracic.heatmap = left_arm_kinematics

    # save trace total rotation value
    participant.right.humerothoracic.trace_total = right_arm_kinematics.cumulative_motion.sum()
    participant.left.humerothoracic.trace_total = left_arm_kinematics.cumulative_motion.sum()

# Format and print results sorted by cumulative time
stats = pstats.Stats(pr)

In [4]:
threshold=0.5  # seconds
long_functions = pstats.Stats(pr) # make a new one

long_functions.stats = {
    func: data
    for func, data in stats.stats.items()
    if data[3] >= threshold  # cumulative time
}

long_functions.sort_stats(pstats.SortKey.TIME).strip_dirs().print_stats()

         33786 function calls (33775 primitive calls) in 15.234 seconds

   Random listing order was used

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        4    6.315    1.579    6.316    1.579 _linalg.py:1666(svd)
        1    0.000    0.000    1.482    1.482 _npyio_impl.py:1118(loadtxt)
        1    0.001    0.001    1.482    1.482 _npyio_impl.py:861(_read)
      648    1.212    0.002    1.212    0.002 kinematics.py:268(_extract_bin_data)
        2    0.005    0.003    3.283    1.642 kinematics.py:570(_populate_heatmap)
      2/1    0.000    0.000    1.482    1.482 data_loading.py:120(load_motion_capture_data)
        8    0.991    0.124    0.991    0.124 _linalg.py:2355(det)
        2    0.000    0.000    4.092    2.046 kinematics.py:81(_get_postural_angles)
        2    0.011    0.006    4.801    2.400 kinematics.py:380(_decompose_rotation_matrices_yxy)
        4    0.585    0.146    1.087    0.272 data_preprocessing.py:143(validate_orthonorm_and_det)

Ok, so the biggest time sinks are:
- `np.linalg.svd()`
- `R.from_matrix()`

These functions seem to be called twice per arm. If we can sink that down to once per arm, we will save up to 3.2 seconds per file (~26% total time). 


- `extract_bin_data` creates the same mask twice per arm, which could be brought down to once, but since this is created using vectorized numpy arrays, it's already super fast and not likely to make a huge difference.
- ``validate_orthonorm_and_det` and `np.linalg.det()` are called frequently, but it's a pretty cheap check. I will skim through and see if we can use it more sparingly (i.e., only after data loading / heavy manipulations), but I'm not super concerned about it.